In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import shapely as sp
import geopandas as gpd

import cartopy.crs as ccrs
import matplotlib.pyplot as plt

from json import loads, dumps
from pyproj import CRS
from pyproj.crs.datum import CustomDatum, CustomPrimeMeridian
from rioxarray.merge import merge_arrays, merge_datasets

In [2]:
def new_crs():
    crs = CRS.from_epsg(4326).to_json()
    crs_json = loads(crs)

    datum = CustomDatum(prime_meridian=CustomPrimeMeridian(18)).to_json()
    datum_json = loads(datum)
    
    custom_crs_json = crs_json
    custom_crs_json['datum'] = datum_json
    custom_crs_json = dumps(custom_crs_json)
    custom_crs = CRS.from_json(custom_crs_json)
    
    return custom_crs

In [3]:
path_east = './weather_raw/2021/humidity/east/'
path_west = './weather_raw/2021/humidity/west/'

hours = ['06/', '09/', '12/', '15/', '18/']
datasets = []

new_crs = new_crs()

for hour in hours:
    data = []

    files_east = [path_east + hour + fname 
                  for fname in np.sort(os.listdir(path_east + hour)) 
                  if fname.endswith('.nc')]
    
    files_west = [path_west + hour + fname 
                  for fname in np.sort(os.listdir(path_west + hour)) 
                  if fname.endswith('.nc')]

    files = list(zip(files_east, files_west))
    
    for file in files:
        east = xr.open_dataset(file[0])
        east = east.rio.write_crs(4326)
        east = east.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
        east = east.rio.reproject(new_crs)

        west = xr.open_dataset(file[1])
        west = west.rio.write_crs(4326)
        west = west.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
        west = west.rio.reproject(new_crs)

        #delta = pd.Timedelta(hours=int(hour[:-1]))
        ew = merge_datasets([east, west])
        #ew = ew.assign_coords(time=(ew.time + delta))
        data.append(ew)
        
    datasets.append(xr.concat(data, dim='time'))
    
ds = xr.merge(datasets)

In [4]:
variables = list(ds.data_vars)

darray = np.array([ds[variable].values for variable in variables])
darray = darray.min(axis=0)

ds['Relative_Humidity_min'] = (['time', 'y', 'x'], darray)
ds = ds.drop_vars(variables)

In [5]:
ds

<xarray.Dataset>
Dimensions:                (time: 184, x: 1721, y: 411)
Coordinates:
  * time                   (time) datetime64[ns] 2021-05-01 ... 2021-10-31
  * x                      (x) float64 1.1 1.2 1.3 1.4 ... 172.9 173.0 173.1
  * y                      (y) float64 81.9 81.8 81.7 81.6 ... 41.1 41.0 40.9
    spatial_ref            int64 0
Data variables:
    Relative_Humidity_min  (time, y, x) float32 nan nan nan nan ... nan nan nan
Attributes:
    Conventions:  CF-1.7

In [6]:
ds.to_netcdf('weather/2021/humidity.nc')